In [ ]:
from pathlib import Path
from speechbrain.inference.speaker import SpeakerRecognition
from speechbrain.utils.fetching import LocalStrategy


MODEL_SOURCE = "speechbrain/spkrec-ecapa-voxceleb" 
MODEL_DIRECTORY = Path('/Users/matrix/Desktop/Projects/vas/backend/ml_models/ecapa')

class SpeakerRecognitionModel:
    def __init__(self, model_source=MODEL_SOURCE, model_directory=MODEL_DIRECTORY):
        self.model_source = model_source
        self.model_directory = model_directory
        self.model = SpeakerRecognition.from_hparams(
            source=self.model_source,
            savedir=self.model_directory,
            local_strategy=LocalStrategy.COPY,
        )

    def generate_embedding(self, audio_file_apth):
        signal = self.model.load_audio(audio_file_apth).unsqueeze(0)
        return self.model.encode_batch(signal).squeeze().detach().cpu().numpy().tolist()



SpeakerRecognition(
  (mods): ModuleDict(
    (compute_features): Fbank(
      (compute_STFT): STFT()
      (compute_fbanks): Filterbank()
      (compute_deltas): Deltas()
      (context_window): ContextWindow()
    )
    (mean_var_norm): InputNormalization()
    (embedding_model): ECAPA_TDNN(
      (blocks): ModuleList(
        (0): TDNNBlock(
          (conv): Conv1d(
            (conv): Conv1d(80, 1024, kernel_size=(5,), stride=(1,))
          )
          (activation): ReLU()
          (norm): BatchNorm1d(
            (norm): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          )
          (dropout): Dropout1d(p=0.0, inplace=False)
        )
        (1): SERes2NetBlock(
          (tdnn1): TDNNBlock(
            (conv): Conv1d(
              (conv): Conv1d(1024, 1024, kernel_size=(1,), stride=(1,))
            )
            (activation): ReLU()
            (norm): BatchNorm1d(
              (norm): BatchNorm1d(1024, eps=1e-05, momentu

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from speechbrain.inference.speaker import SpeakerRecognition
from speechbrain.utils.fetching import LocalStrategy 


MODEL_SOURCE = "speechbrain/spkrec-ecapa-voxceleb" 
MODEL_DIRECTORY = Path('/Users/matrix/Desktop/Projects/vas/backend/ml_models/ecapa')
DATAFRAME_PATH = Path(
    "/Users/matrix/Desktop/Projects/vas/"
    "research_data/vas_enrollment_2026/"
    "private_ecapa_embedding_dataframe.pkl"
)

df = pd.read_pickle(DATAFRAME_PATH)
print(df.head(5))



   sample_id participant_id  user_id  user         email  speech_type  \
0          1           P001        2  dk_p  dk@gmail.com  free_speech   
1          2           P001        2  dk_p  dk@gmail.com  free_speech   
2          3           P001        2  dk_p  dk@gmail.com  free_speech   
3          4           P001        2  dk_p  dk@gmail.com  free_speech   
4          5           P001        2  dk_p  dk@gmail.com  free_speech   

   duration_seconds                      audio_local_path  \
0              2.76  audio/P001/free_speech/sample-1.webm   
1              2.64  audio/P001/free_speech/sample-2.webm   
2              1.74  audio/P001/free_speech/sample-3.webm   
3              1.98  audio/P001/free_speech/sample-4.webm   
4              2.10  audio/P001/free_speech/sample-5.webm   

                                           embedding  embedding_dimension  \
0  [36.932297, 6.3530836, 23.224821, -8.24955, -3...                  192   
1  [-34.498642, 6.0060077, -19.98251, 27

In [11]:
df.columns

Index(['sample_id', 'participant_id', 'user_id', 'user', 'email',
       'speech_type', 'duration_seconds', 'audio_local_path', 'embedding',
       'embedding_dimension', 'model_version'],
      dtype='str')

In [14]:
filtered_df = df[['user_id', 'speech_type', 'duration_seconds','embedding'] ]


In [15]:
df.speech_type.value_counts()

speech_type
free_speech    160
pangrams       140
Name: count, dtype: int64

In [24]:
free_speech = filtered_df[filtered_df['speech_type'] == 'free_speech']
pangrams = filtered_df[filtered_df['speech_type'] == 'pangrams']

In [25]:
free_speech.speech_type.value_counts()

speech_type
free_speech    160
Name: count, dtype: int64

In [26]:
pangrams.speech_type.value_counts()

speech_type
pangrams    140
Name: count, dtype: int64

In [51]:
free_speech



,user_id,speech_type,duration_seconds,embedding
0,2,free_speech,2.760000,"[36.932297, 6.3530836, 23.224821, -8.24955, -3..."
1,2,free_speech,2.640000,"[-34.498642, 6.0060077, -19.98251, 27.459469, ..."
2,2,free_speech,1.740000,"[-35.293575, -13.314312, -29.885775, -1.690824..."
3,2,free_speech,1.980000,"[-33.71563, -8.27705, -9.514201, 1.2937112, -1..."
4,2,free_speech,2.100000,"[-32.300438, 6.6385546, -6.791016, 12.18355, 3..."
...,...,...,...,...
285,26,free_speech,8.020000,"[37.28882, -28.803705, 10.217835, -3.4393048, ..."
286,26,free_speech,2.644000,"[36.468834, -3.106622, 34.448856, 16.008673, 2..."
287,26,free_speech,7.849312,"[2.2211802, 4.9810696, 9.129651, -28.237915, 1..."
288,26,free_speech,6.932000,"[12.426937, -18.931484, 8.229936, -7.2015843, ..."


In [39]:
import numpy as np

rng = np.random.default_rng(seed=42)

free_speech_split = {}

for user_id, rows in free_speech.groupby('user_id'):
    enroll_index = rng.choice(
        rows.index.to_numpy(),
        size=6,
        replace=False
    )
    free_speech_split[user_id] = {
        'enroll': rows.loc[enroll_index],
        'test': rows.drop(enroll_index)
    }

In [55]:
free_speech_split[2]['enroll'].embedding

6    [-28.50908, -7.521462, -12.673597, 13.815689, ...
4    [-32.300438, 6.6385546, -6.791016, 12.18355, 3...
8    [-57.89734, 3.1213057, -5.817234, -6.2331886, ...
9    [5.451133, -0.24065414, -4.997326, -1.2285507,...
3    [-33.71563, -8.27705, -9.514201, 1.2937112, -1...
0    [36.932297, 6.3530836, 23.224821, -8.24955, -3...
Name: embedding, dtype: object

In [57]:
free_speech_mean_embedding = {}

for user_id, embeddings in free_speech_split.items():
    stacked_embeddings = np.vstack(embeddings['enroll'].embedding)
    centroid = stacked_embeddings.mean(axis=0)
    free_speech_mean_embedding[user_id] = centroid

In [58]:
free_speech_mean_embedding

{2: array([-1.83398418e+01,  1.22962790e-02, -2.76142526e+00,  1.93027687e+00,
         7.23459196e+00, -2.18880596e+01, -1.79407711e+01,  1.36100426e+01,
        -3.03570461e+01,  1.38021460e+01,  3.81693196e+00,  1.31826248e+01,
         2.59092593e+00,  1.18807297e+01, -1.29825974e+01,  1.38987007e+01,
         2.90071487e+00, -3.28279343e+01,  8.93642724e-01, -1.76622620e+01,
        -6.26015663e-01, -7.73489904e+00, -2.54804001e+01, -8.51524258e+00,
        -7.20271444e+00, -1.45851049e+01, -1.46178484e+01,  3.92184901e+00,
         7.37965059e+00,  2.18079243e+01,  1.87702618e+01,  2.23169494e+00,
        -6.60406542e+00,  2.31528111e+01,  1.84743862e+01, -9.87264156e+00,
        -4.55557489e+00, -1.72960835e+01,  3.62710166e+00,  1.49554176e+01,
         9.22028637e+00, -1.46928978e+01,  3.68013992e+01, -1.14652901e+01,
         5.53144503e+00, -2.53747135e-01,  1.49453430e+01,  1.40560656e+01,
         1.75505600e+01, -1.49723063e+01, -4.32098913e+00, -7.39697266e+00,
        -

In [59]:
def cosine_similarity(vec1, vec2):
    numerator = np.dot(vec1, vec2)
    denominator = np.linalg.norm(vec1) * np.linalg.norm(vec2)
    return numerator / denominator



In [62]:
for user_id, centroid in free_speech_mean_embedding.items():

    for test_embedding in free_speech_split[user_id]['enroll'].embedding:
        similarity = cosine_similarity(centroid, test_embedding)
        
        print(f"User {user_id} similarity: {similarity:.4f}")

User 2 similarity: 0.8491
User 2 similarity: 0.7628
User 2 similarity: 0.7831
User 2 similarity: 0.7164
User 2 similarity: 0.7675
User 2 similarity: 0.4329
User 4 similarity: 0.8285
User 4 similarity: 0.8081
User 4 similarity: 0.8318
User 4 similarity: 0.7491
User 4 similarity: 0.6770
User 4 similarity: 0.3692
User 5 similarity: 0.4494
User 5 similarity: 0.6239
User 5 similarity: 0.7565
User 5 similarity: 0.7090
User 5 similarity: 0.7362
User 5 similarity: 0.6956
User 7 similarity: 0.9021
User 7 similarity: 0.2966
User 7 similarity: 0.8417
User 7 similarity: 0.7981
User 7 similarity: 0.8550
User 7 similarity: 0.9001
User 8 similarity: 0.6349
User 8 similarity: 0.7410
User 8 similarity: 0.7121
User 8 similarity: 0.7482
User 8 similarity: 0.7326
User 8 similarity: 0.5537
User 9 similarity: 0.9242
User 9 similarity: 0.8746
User 9 similarity: 0.9554
User 9 similarity: 0.9575
User 9 similarity: 0.9588
User 9 similarity: 0.9614
User 10 similarity: 0.9229
User 10 similarity: 0.8286
User 10 si